[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmcguir2/astromol/blob/refactor/docs/notebooks/01_quickstart.ipynb)

# astromol quickstart

This notebook loads the production database, creates census views, and performs a few basic queries.

In [ ]:
# Run this setup cell first in Google Colab. In a local checkout with astromol
# already installed, it does nothing.
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/bmcguir2/astromol.git@refactor",
    ])

In [ ]:
from astromol.census import CensusView
from astromol.database import Database

db = Database()

print(f"references: {len(db.refs)}")
print(f"telescopes: {len(db.telescopes)}")
print(f"sources: {len(db.sources)}")
print(f"molecules: {len(db.molecules)}")
print(f"detections: {len(db.detections)}")

## Create census views

Use a frozen census view for historical reproduction and `current` for the live database.

In [ ]:
view_2021 = CensusView.for_census(db, "2021")
view_2026 = CensusView.for_census(db, "2026")
current = CensusView.current(db)

for label, view in [("2021", view_2021), ("2026", view_2026), ("current", current)]:
    print(label, len(view.ism_molecules()), "ISM/CSM molecules")

## Query a context

Context views exclude isotopologues by default. Pass `include_isotopologues=True` when you want the expanded isotope inventory.

In [ ]:
ppd_standard = view_2026.ppd_molecules()
ppd_with_isotopologues = view_2026.ppd_molecules(include_isotopologues=True)

print("PPD molecules, standard view:", len(ppd_standard))
print("PPD molecules, isotope-expanded view:", len(ppd_with_isotopologues))

ppd_with_isotopologues[:10]

In [ ]:
# Inspect one molecule and its detections.
molecule = db.molecules["mol:CH3OH"]
print(molecule.name)
print(molecule.formula)
print(molecule.mass)

for detection in db.detections:
    if detection.molecule.label == molecule.label:
        print(detection.id, detection.type, detection.status, detection.year)